## Import Dependencies

In [ ]:
from transformers import AutoTokenizer
import polars as pl
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset

## Import Data

In [ ]:
train_path = "/content/complete_train.parquet"
test_path = "/content/complete_test.parquet"

In [ ]:
train_df = pl.scan_parquet(train_path)
test_df = pl.scan_parquet(test_path)

In [ ]:
test_df.collect()

id,asym_id,amino_acid,index,strand_id,secondary_structure,rcsb_id,sequence,length
str,str,str,i64,str,str,str,str,i64
"""3VKW""","""A""","""SER""",1,"""A""",""" ""","""3VKW_1""","""SYTRSEEIESLEQFHMATASSLIHKQMCSI…",446
"""3VKW""","""A""","""TYR""",2,"""A""",""" ""","""3VKW_1""","""SYTRSEEIESLEQFHMATASSLIHKQMCSI…",446
"""3VKW""","""A""","""THR""",3,"""A""",""" ""","""3VKW_1""","""SYTRSEEIESLEQFHMATASSLIHKQMCSI…",446
"""3VKW""","""A""","""ARG""",4,"""A""",""" ""","""3VKW_1""","""SYTRSEEIESLEQFHMATASSLIHKQMCSI…",446
"""3VKW""","""A""","""SER""",5,"""A""",""" ""","""3VKW_1""","""SYTRSEEIESLEQFHMATASSLIHKQMCSI…",446
…,…,…,…,…,…,…,…,…
"""5XVT""","""A""","""LEU""",693,"""A""","""T""","""5XVT_1""","""MGSSHHHHHHSSGLVPRGSHMSSVDQKAIS…",697
"""5XVT""","""A""","""ARG""",694,"""A""",""".""","""5XVT_1""","""MGSSHHHHHHSSGLVPRGSHMSSVDQKAIS…",697
"""5XVT""","""A""","""SER""",695,"""A""",""".""","""5XVT_1""","""MGSSHHHHHHSSGLVPRGSHMSSVDQKAIS…",697


#### Removing Duplicates on Index
Some indices have two amino acids, so remove the one without a secondary structure

In [ ]:
train_removed = (
    train_df.sort(['id', 'asym_id', 'index', 'secondary_structure'])
            .group_by(['id', 'asym_id', 'index'])
            .agg(['amino_acid', 'secondary_structure'])
            .filter((pl.col('amino_acid').list.len() > 1))
            .with_columns(
                amino_acid = pl.col('amino_acid').list.slice(1),
                secondary_structure = pl.col('secondary_structure').list.slice(1),
            )
            .explode(['amino_acid', 'secondary_structure'])
)

train_df = (
    train_df.join(train_removed, on=['id', 'asym_id', 'index'], how='left')
            .with_columns(
                          amino_acid = pl.when(~(pl.col('amino_acid_right').is_null()))
                                        .then(pl.col('amino_acid_right'))
                                        .otherwise(pl.col('amino_acid')),
                         secondary_structure = pl.when(~(pl.col('secondary_structure_right').is_null()))
                                        .then(pl.col('secondary_structure_right'))
                                        .otherwise(pl.col('secondary_structure'))
            )
            .drop(['amino_acid_right', 'secondary_structure_right'])
            .unique(['id', 'asym_id', 'index'])
            .sort(['id', 'asym_id', 'index'])
)


In [ ]:
test_removed = (
    test_df.sort(['id', 'asym_id', 'index', 'secondary_structure'])
            .group_by(['id', 'asym_id', 'index'])
            .agg(['amino_acid', 'secondary_structure'])
            .filter((pl.col('amino_acid').list.len() > 1))
            .with_columns(
                amino_acid = pl.col('amino_acid').list.slice(1),
                secondary_structure = pl.col('secondary_structure').list.slice(1),
            )
            .explode(['amino_acid', 'secondary_structure'])
)

test_df = (
    test_df.join(test_removed, on=['id', 'asym_id', 'index'], how='left')
            .with_columns(
                          amino_acid = pl.when(~(pl.col('amino_acid_right').is_null()))
                                        .then(pl.col('amino_acid_right'))
                                        .otherwise(pl.col('amino_acid')),
                         secondary_structure = pl.when(~(pl.col('secondary_structure_right').is_null()))
                                        .then(pl.col('secondary_structure_right'))
                                        .otherwise(pl.col('secondary_structure')),
            )
            .drop(['amino_acid_right', 'secondary_structure_right'])
            .unique(['id', 'asym_id', 'index'])
            .sort(['id', 'asym_id', 'index'])
)

In [ ]:
test_df.collect()

id,asym_id,amino_acid,index,strand_id,secondary_structure,rcsb_id,sequence,length
str,str,str,i64,str,str,str,str,i64
"""1A1X""","""A""","""GLY""",1,"""A""",""" ""","""1A1X_1""","""GSAGEDVGAPPDHLWVHQEGIYRDEYQRTW…",108
"""1A1X""","""A""","""SER""",2,"""A""",""" ""","""1A1X_1""","""GSAGEDVGAPPDHLWVHQEGIYRDEYQRTW…",108
"""1A1X""","""A""","""ALA""",3,"""A""",""".""","""1A1X_1""","""GSAGEDVGAPPDHLWVHQEGIYRDEYQRTW…",108
"""1A1X""","""A""","""GLY""",4,"""A""",""".""","""1A1X_1""","""GSAGEDVGAPPDHLWVHQEGIYRDEYQRTW…",108
"""1A1X""","""A""","""GLU""",5,"""A""",""".""","""1A1X_1""","""GSAGEDVGAPPDHLWVHQEGIYRDEYQRTW…",108
…,…,…,…,…,…,…,…,…
"""7ODC""","""A""","""ILE""",420,"""A""",""" ""","""7ODC_1""","""MSSFTKDEFDCHILDEGFTAKDILDQKINE…",424
"""7ODC""","""A""","""GLN""",421,"""A""",""" ""","""7ODC_1""","""MSSFTKDEFDCHILDEGFTAKDILDQKINE…",424
"""7ODC""","""A""","""SER""",422,"""A""",""" ""","""7ODC_1""","""MSSFTKDEFDCHILDEGFTAKDILDQKINE…",424


## Process Data

### Add Labels

In [ ]:
le = LabelEncoder()

train_df = (train_df
            .with_columns(label =
                          le.fit_transform(
                              train_df.select(['secondary_structure'])
                                      .collect()['secondary_structure']))
            )

test_df = (test_df
            .with_columns(label =
                          le.transform(
                              test_df.select(['secondary_structure'])
                                      .collect()['secondary_structure']))
          )

### Prepare for Tokenization

In [ ]:
train_agg = (train_df
              .select(['id', 'asym_id', 'sequence', 'index', 'label'])
              .group_by(['id', 'asym_id', 'sequence'])
              .agg(['index', 'label'])
            )

test_agg = (test_df
              .select(['id', 'asym_id', 'sequence', 'index', 'label'])
              .group_by(['id', 'asym_id', 'sequence'])
              .agg(['index', 'label'])
            )

In [ ]:
train_ds = Dataset.from_polars(train_agg.collect())
test_ds = Dataset.from_polars(test_agg.collect())

### Tokenization

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")

def tokenize_and_label(examples):
  tokenized_inputs = tokenizer(examples["sequence"],
                                return_tensors="pt",
                                add_special_tokens=False,
                                padding=False)
  tokenized_inputs["input_ids"] = tokenized_inputs["input_ids"][0]
  tokenized_inputs["attention_mask"] = tokenized_inputs["attention_mask"][0]
  return tokenized_inputs

In [ ]:
tokenized_train = train_ds.map(tokenize_and_label, batched=False)
tokenized_test = test_ds.map(tokenize_and_label, batched=False)

Map:   0%|          | 0/11707 [00:00<?, ? examples/s]

Map:   0%|          | 0/2974 [00:00<?, ? examples/s]

In [ ]:
import gc
gc.collect()

0

## Output Data

### Dataset to DataFrame

In [ ]:
output_train = Dataset.to_polars(tokenized_train).lazy()
output_test = Dataset.to_polars(tokenized_test).lazy()

In [ ]:
output_train = (output_train
                .join(train_agg,
                      on=['id', 'asym_id', 'sequence', 'index', 'label'],
                      how='inner')
                .explode(['index', 'label', 'input_ids', 'attention_mask'])
                )

output_test = (output_test
                .join(test_agg,
                      on=['id', 'asym_id', 'sequence', 'index', 'label'],
                      how='inner')
                .explode(['index', 'label', 'input_ids', 'attention_mask'])
                )


### Write to Parquet

In [ ]:
output_train.collect().write_parquet('tokenized_train.parquet')
output_test.collect().write_parquet('tokenized_test.parquet')